# IS630: Green Light for an Indie Studio
## Notebook 00: Setup, Acquisition, Join and Train/Test Split
**Group**: [Insert names]

> **AI Coding Assistant Declaration**:
> In accordance with course guidelines, an AI coding assistant was used strictly for code scaffolding and data preparation. All code blocks below are labeled as AI-assisted. All modelling decisions, interpretations, and written commentaries are independently authored by group members in the designated sections below.

| Dataset | Kaggle ref | Version | Pulled |
|---|---|---|---|
| Steam | `fronkongames/steam-games-dataset` | v49 | 19 Sep 2026 |
| HowLongToBeat | `b4n4n4p0wer/how-long-to-beat-video-game-playtime-dataset` | v5 | 19 Sep 2026 |

The Steam dataset refreshes continuously. Do not re-download mid-project.

**Reads:** `data/raw/steam_v49.zip`, `data/raw/hltb_v5.zip`
**Writes:** `00_steam.parquet`, `00_hltb.parquet`, `00_joined.parquet`, `00_train.parquet`, `00_test_holdout.parquet`, all in `data/interim`

## 1. Environment setup and data acquisition
*Code: Generated by AI Assistant*

In [ ]:
import sys
import urllib.request
import zipfile
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").exists()) / "src"))
from is630 import config
from is630.clean import normalise_title

TEST_FRACTION = 0.20

In [ ]:
# Download only if the pinned archive is missing
for path, ref in {config.STEAM_ZIP: config.STEAM_REF, config.HLTB_ZIP: config.HLTB_REF}.items():
    if not path.exists():
        urllib.request.urlretrieve(f"https://www.kaggle.com/api/v1/datasets/download/{ref}", path)
    print(path.name, f"{path.stat().st_size / 1e6:.1f} MB")

## 2. Steam dataset
*Code: Generated by AI Assistant*

*Written by AI Assistant*: the CSV header fuses `Discount` and `DLC count` into one name, giving 39 headers for 40 fields. Reading it as-is shifts every column one place, so the header is split before loading.

In [ ]:
with zipfile.ZipFile(config.STEAM_ZIP) as z:
    with z.open("games.csv") as f:
        header = pd.read_csv(f, nrows=0).columns.tolist()

i = header.index("DiscountDLC count")
names = header[:i] + ["Discount", "DLC count"] + header[i + 1:]

with zipfile.ZipFile(config.STEAM_ZIP) as z:
    with z.open("games.csv") as f:
        steam = pd.read_csv(f, header=0, names=names, index_col=False)

print(f"steam: {steam.shape[0]:,} rows, {steam.shape[1]} columns")
steam.head()

In [ ]:
print("Missing values, columns with any:")
print(steam.isna().sum()[lambda v: v > 0].sort_values(ascending=False))
print("\nZero-filled rather than null:")
for col in ["Metacritic score", "Price", "Positive", "Negative", "Median playtime forever"]:
    print(f"  {col}: {(steam[col] == 0).sum():,} zeros")

### Interpretation: Steam data inspection
**Author**: [Insert Name]  
<!-- Record your observations on the Steam table: row and column counts, which columns are missing values and whether that missingness looks random, which columns are zero-filled rather than null and what that hides, and which columns the three analytical questions will actually need. -->


## 3. HowLongToBeat dataset
*Code: Generated by AI Assistant*

In [ ]:
with zipfile.ZipFile(config.HLTB_ZIP) as z:
    with z.open("hltb_dataset_normalized.csv") as f:
        hltb = pd.read_csv(f)

print(f"hltb: {hltb.shape[0]:,} rows, {hltb.shape[1]} columns")
hltb.head()

In [ ]:
print("Entry types:")
print(hltb["type"].value_counts())
print(f"\nmain_story missing: {hltb['main_story'].isna().sum():,} of {len(hltb):,}")
print(f"release_year missing: {hltb['release_year'].isna().sum():,} of {len(hltb):,}")

### Interpretation: HowLongToBeat data inspection
**Author**: [Insert Name]  
<!-- Record your observations on the HowLongToBeat table: how many entries are games rather than DLC or other types, how much of the table carries a main-story time, and what that implies for how many Steam games can receive a completion ratio. -->


## 4. Save the raw tables
*Code: Generated by AI Assistant*

*Written by AI Assistant*: both tables are saved as loaded. Column pruning and type cleaning belong to later stages.

In [ ]:
steam.to_parquet(config.INTERIM / "00_steam.parquet", index=False)
hltb.to_parquet(config.INTERIM / "00_hltb.parquet", index=False)
print(f"saved unmodified: {len(steam):,} steam rows, {len(hltb):,} hltb rows")

## 5. Joining the two datasets
*Code: Generated by AI Assistant*

*Written by AI Assistant*: no shared identifier exists, so the key is the normalised title. Release year is recorded rather than used as part of the key, because HowLongToBeat stores a game's original release year while Steam stores its Steam release date. Requiring the two to agree would drop console-first ports and re-releases, which are among the best covered titles.

Both sides are reduced to one row per title first, so the join cannot multiply rows:

- HowLongToBeat is filtered to `type == "game"` with a positive `main_story`, then for each title the candidate whose release year is closest to the Steam release year is kept, breaking ties on the larger `main_story_polled`.
- Steam keeps, per title, the entry with the most reviews.

The join is a left join, so every Steam game survives and `matched_hltb` marks the ones that found a partner. `year_diff` is the Steam year minus the HowLongToBeat year, for tightening the match later.

In [ ]:
steam["title_key"] = normalise_title(steam["Name"])
steam["steam_year"] = pd.to_datetime(steam["Release date"], errors="coerce", format="mixed").dt.year

hltb["title_key"] = normalise_title(hltb["name"])

candidates = hltb[hltb["type"].eq("game") & hltb["main_story"].notna() & hltb["main_story"].gt(0)].copy()
print(f"hltb candidates with a main story time: {len(candidates):,}")

steam_one = (
    steam.assign(_reviews=steam["Positive"] + steam["Negative"])
    .sort_values("_reviews", ascending=False)
    .drop_duplicates("title_key")
    .drop(columns="_reviews")
)
print(f"steam rows dropped as duplicate titles: {len(steam) - len(steam_one):,}")

In [ ]:
# Pick, per title, the hltb candidate closest in year to the Steam release
pairs = steam_one[["title_key", "steam_year"]].merge(candidates, on="title_key", how="inner")
pairs["_gap"] = (pairs["steam_year"] - pairs["release_year"]).abs()

hltb_one = (
    pairs.sort_values(["_gap", "main_story_polled"], ascending=[True, False])
    .drop_duplicates("title_key")
    .drop(columns=["_gap", "steam_year"])
)

joined = steam_one.merge(hltb_one, on="title_key", how="left", suffixes=("", "_hltb"))
joined["matched_hltb"] = joined["main_story"].notna()
joined["year_diff"] = joined["steam_year"] - joined["release_year"]

assert len(joined) == len(steam_one), "join multiplied rows"
matched = int(joined["matched_hltb"].sum())
agree = int((joined["year_diff"].abs() <= 1).sum())
print(f"rows in: {len(steam_one):,}")
print(f"matched to hltb: {matched:,} ({matched / len(joined) * 100:.1f}%)")
print(f"  of which year agrees within 1: {agree:,} ({agree / matched * 100:.1f}% of matches)")
print(f"rows out: {len(joined):,}")

In [ ]:
# Matched against unmatched, on the variables both groups have
compare = joined.groupby("matched_hltb").agg(
    games=("AppID", "size"),
    median_reviews=("Positive", "median"),
    median_price=("Price", "median"),
    pct_with_metacritic=("Metacritic score", lambda s: (s > 0).mean() * 100),
    median_year=("steam_year", "median"),
)
compare.round(2)

In [ ]:
joined.to_parquet(config.INTERIM / "00_joined.parquet", index=False)
joined[["AppID", "Name", "steam_year", "release_year", "year_diff", "main_story", "main_story_polled", "matched_hltb"]].head()

### Interpretation: The join and its match rate
**Author**: [Insert Name]  
<!-- Interpret the match rate and the matched-versus-unmatched comparison above. Cover whether the matched subset looks representative of the Steam population, what the direction of any difference implies for the completion ratio results, whether the normalised-title key should be tightened to require the release years to agree, and how this should be reported as a limitation. -->


## 6. Train and test split
*Code: Generated by AI Assistant*

*Written by AI Assistant*: the test set is held out now, before any exploration, and is scored once at the end. Nothing is refitted after that score.

The split is stratified on release year crossed with `matched_hltb`, so both sets span the same years and carry the same share of matched games. A plain random split would leave the match rate and the age mix differing between them, and review counts and playtime both accumulate with age.

A time-based holdout, training on older games and testing on the newest, would be the stricter test of predicting an unreleased game. It is not used here because every accumulating column would then differ systematically between train and test, which would confound the score.

In [ ]:
strata = joined["matched_hltb"].astype(str) + "_" + joined["steam_year"].fillna(-1).astype(int).astype(str)
rare = strata.value_counts()[lambda v: v < 5].index
strata = strata.where(~strata.isin(rare), "rare")

test = joined.groupby(strata, group_keys=False).sample(frac=TEST_FRACTION, random_state=config.SEED)
train = joined.drop(index=test.index)

assert len(train) + len(test) == len(joined)
assert not set(train["AppID"]) & set(test["AppID"])

for name, df in [("00_train", train), ("00_test_holdout", test)]:
    df.to_parquet(config.INTERIM / f"{name}.parquet", index=False)
    print(f"{name}: {len(df):,} rows ({len(df) / len(joined) * 100:.1f}%), matched {df['matched_hltb'].mean() * 100:.1f}%")

### Interpretation: Split justification and decisions
**Author**: [Insert Name]  
<!-- Give your reasoning for the split design. Cover why stratified random was chosen over a chronological holdout for this data, whether you accept that argument or want the time-based version instead, why the test set is held out before any exploration, and what the 20 percent size means for the precision of the final score. -->


*Written by AI Assistant*: `00_test_holdout.parquet` must not be loaded by any notebook except the final scoring step. Every later stage starts from `00_train.parquet`.